In [13]:
import mlflow

mlflow.set_tracking_uri(f'sqlite:///{db_path}')
mlflow.set_experiment('nyc_greentaxi_2021_tripdata_experiment') 

<Experiment: artifact_location='/Users/mac/Projects/MLops_datatalks/mlruns/2', creation_time=1752919176206, experiment_id='2', last_update_time=1752919176206, lifecycle_stage='active', name='nyc_greentaxi_2021_tripdata_experiment', tags={}>

In [5]:
from mlflow.tracking import MlflowClient

RUN_ID ='ca613be92280409287899eaab84e698f'
client = MlflowClient(tracking_uri="http://localhost:5001")
run = client.get_run(RUN_ID)
print(run.info)

<RunInfo: artifact_uri='/Users/mac/Projects/MLops_datatalks/mlruns/2/ca613be92280409287899eaab84e698f/artifacts', end_time=1752920764917, experiment_id='2', lifecycle_stage='active', run_id='ca613be92280409287899eaab84e698f', run_name='upbeat-lamb-335', run_uuid='ca613be92280409287899eaab84e698f', start_time=1752920722598, status='FINISHED', user_id='mac'>


In [4]:
#path = client.download_artifacts(run_id='ca613be92280409287899eaab84e698f', path='dict_vectorizer.bin')
client.list_artifacts(RUN_ID)

[<FileInfo: file_size=None, is_dir=True, path='models_xgboost_mlflow'>,
 <FileInfo: file_size=None, is_dir=True, path='preprocessor'>]

In [14]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = (df.lpep_dropoff_datetime - df.lpep_pickup_datetime)
    df['duration'] = df['duration'].apply(lambda td: td.total_seconds()/60)
    df = df[((df['duration'] > 1) & (df['duration'] <= 60))]
    
    categorical = ['PULocationID','DOLocationID']
    df[categorical] = df[categorical].astype(str)

    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    return df

In [15]:
df_train = read_dataframe('green_tripdata/green_tripdata_2021-01.parquet')
df_val = read_dataframe('green_tripdata/green_tripdata_2021-02.parquet')

In [16]:
categorical = ['PU_DO']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
val_dicts = df_val[categorical + numerical].to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

y_train = df_train['duration'].values
y_val = df_val['duration'].values

In [17]:
from math import sqrt

In [18]:
import xgboost as xgb

In [19]:
import mlflow.xgboost
from mlflow.models.signature import infer_signature
mlflow.xgboost.autolog(disable=True)

In [ ]:
with mlflow.start_run():
    train = xgb.DMatrix(X_train, label=y_train)
    val = xgb.DMatrix(X_val, label=y_val)

    mlflow.set_tag("engineer", "richkinwe")
    mlflow.log_param("train_data_path", "green_tripdata/green_tripdata_2021-01.parquet")
    mlflow.log_param("val_data_path", "green_tripdata/green_tripdata_2021-02.parquet")
    mlflow.set_tag("model", "xgboost")
    
    best_params = {
    'max_depth': 30,
    'learning_rate': 0.0959,
    'reg_alpha': 0.0181,
    'reg_lambda': 0.0117,
    'min_child_weight': 1.0606,
    'objective': 'reg:linear',
    'seed': 42
 }
    
    mlflow.log_params(best_params)

    booster = xgb.train(
            params=best_params,
            dtrain=train,
            num_boost_round=50,
            evals=[(val, 'validation')],
            early_stopping_rounds=50
        )
    
    y_pred = booster.predict(val)
    rmse = sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)

    with open('models.preprocessor.b', 'wb') as f_out:
        pickle.dump(dv, f_out)

    mlflow.log_artifact('models.preprocessor.b', artifact_path='preprocessor')

    mlflow.xgboost.log_model(booster, artifact_path='models_xgboost_mlflow')